# AmsterdamUMCdb Exploration

## Capstone Goal
Predict successful weaning from mechanical ventilation using time-series deep learning models.

## Objectives
- Understand database structure
- Identify ventilation-related tables
- Find extubation events
- Explore physiological measurements
- Understand timestamps and patient identifiers

### Imports

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
import matplotlib.pyplot as plt

### Retrieving Google Project Id

In [ ]:
client = bigquery.Client()

print("Connected to BigQuery")

In [ ]:
# sets *your* project id
PROJECT_ID = "capstoneweaningprediction" #@param {type:"string"}

# Sets the default BigQuery dataset for accessing AmsterdamUMCdb

If you have received instructions to use a specific BigQuery instance, change the default settings here. Otherwise use these default values.

In [ ]:
# sets default dataset for AmsterdamUMCdb
DATASET_PROJECT_ID = 'amsterdamumcdb' #@param {type:"string"}
DATASET_ID = 'version1_5_0' #@param {type:"string"}
LOCATION = 'eu' #@param {type:"string"}

# Provide your credentials to access the AmsterdamUMCdb dataset on Google BigQuery
Authenticate your credentials with Google Cloud Platform and set your default Google Cloud Project ID as an environment variable for running query jobs.

1. Run the cell. The `Allow this notebook to access your Google credentials?` prompt appears. Select `Allow`.
2. In the `Sign in - Google Accounts` dialog, use the account you registered during the AmsterdamUMCdb application process and select `Allow` again.

In [ ]:
import os
from google.colab import auth

# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

auth.authenticate_user()
print('Authenticated')

# Enable data table display

Colab includes the `google.colab.data_table` package that can be used to display Pandas dataframes as an interactive data table (default limits: `max_rows = 20000`, `max_columns = 20`). This is especially useful when exploring the  tables or dictionary from AmsterdamUMCdb. It can be enabled with:

In [ ]:
%load_ext google.colab.data_table
from google.colab.data_table import DataTable

# change default limits:
DataTable.max_columns = 50
DataTable.max_rows = 80000


## Set the default query job configuration for magics

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

## Query the `person` table and copy the data to the `persons` Pandas dataframe:

The `person` table contains a record for each patient in AmsterdamUMCdb.

Since this is a relatively small table, it is acceptable to use `SELECT *`.

**Note**: Should an error occur while running the query, please see
the AmsterdamUMCdb BigQuery [Frequently Asked Questions](https://github.com/AmsterdamUMC/AmsterdamUMCdb/wiki/bigquery#faq).

In [ ]:
%%bigquery person
SELECT * FROM `amsterdamumcdb.version1_5_0.person`;


## Set the default query job configuration for google-cloud-bigquery client

In [ ]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

## 1. Define the Invasive Mechanical Ventilation concept set

From the concept exploration step (`Test__2_.ipynb`, cell 28), eight procedure concepts represent **invasive** mechanical ventilation modes in AmsterdamUMCdb. We deliberately exclude non-invasive modes (NIPPV, CPAP) to match Zappalà's IMV definition.

| Concept ID | Name | Rationale |
|---|---|---|
| 4230167 | Artificial ventilation (umbrella) | Highest patient count (14,807); covers all IMV |
| 37154096 | Pressure support ventilation | Most common mode in AmsterdamUMCdb |
| 37151337 | Continuous mandatory ventilation pressure-controlled | Common IMV mode |
| 37152413 | Continuous mandatory ventilation volume-controlled | Common IMV mode |
| 37152364 | Synchronized intermittent mandatory ventilation (variant) | Less common IMV mode |
| 4055375 | Assisted controlled mandatory ventilation | Less common IMV mode |
| 37151339 | Continuous mandatory ventilation volume-targeted | Rare IMV variant |
| 37153603 | Continuous mandatory ventilation | Rare IMV variant |

**Excluded** (non-invasive, used in step 5 to identify NIV episodes for context):
- 40486624 Non-invasive positive pressure ventilation
- 4165535 Continuous positive airway pressure ventilation

In [ ]:
IMV_CONCEPTS = [
    4230167,    # Artificial ventilation (umbrella)
    37154096,   # Pressure support ventilation
    37151337,   # CMV pressure-controlled
    37152413,   # CMV volume-controlled
    37152364,   # SIMV (variant)
    4055375,    # Assisted controlled mandatory ventilation
    37151339,   # CMV volume-targeted
    37153603,   # CMV (generic)
]
NIV_CONCEPTS = [40486624, 4165535]   # non-invasive — for context only

IMV_CONCEPTS_SQL = ','.join(str(c) for c in IMV_CONCEPTS)
print(f'{len(IMV_CONCEPTS)} IMV concept IDs defined')

## 2. Extract all Invasive Mechanical Ventilation logs

Pull every IMV log (one row per hourly ventilator entry) for every patient in the database. This will be tens of millions of rows, but we only keep the columns we need.

In [ ]:
query = f"""
SELECT
    po.person_id,
    po.visit_occurrence_id,
    po.procedure_concept_id,
    po.procedure_datetime,
    po.procedure_end_datetime
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` po
WHERE (po.provider_id IS NOT NULL OR po.provider_id IS NULL)
  AND po.procedure_concept_id IN ({IMV_CONCEPTS_SQL})
ORDER BY po.visit_occurrence_id, po.procedure_datetime
"""

imv_logs = client.query(query).to_dataframe()
print(f'Loaded {len(imv_logs):,} IMV logs across {imv_logs["visit_occurrence_id"].nunique():,} visits')
imv_logs.head()

## 3. Group consecutive logs into episodes (the 48-hour rule)

Following Zappalà 2025: consecutive Invasive Mechanical Ventilation logs that are **less than 48 hours apart** belong to the same ventilation episode. A gap of 48h or more starts a new episode (or, if at the end of the record, signals successful weaning).

Implementation: sort logs within each visit by time, compute the gap to the previous log, and start a new episode whenever the gap is ≥ 48h.

In [ ]:
# Sort and compute gap to previous log within each visit
imv_logs = imv_logs.sort_values(['visit_occurrence_id', 'procedure_datetime']).reset_index(drop=True)
imv_logs['prev_datetime'] = imv_logs.groupby('visit_occurrence_id')['procedure_datetime'].shift(1)
imv_logs['gap_hours'] = (
    imv_logs['procedure_datetime'] - imv_logs['prev_datetime']
).dt.total_seconds() / 3600

# Start of a new episode = first log of a visit OR gap >= 48h
imv_logs['new_episode'] = imv_logs['gap_hours'].isna() | (imv_logs['gap_hours'] >= 48)

# Episode index per visit
imv_logs['episode_idx'] = imv_logs.groupby('visit_occurrence_id')['new_episode'].cumsum()

print(f'Logs:    {len(imv_logs):,}')
print(f'Visits:  {imv_logs["visit_occurrence_id"].nunique():,}')
imv_logs[['visit_occurrence_id', 'procedure_datetime', 'gap_hours', 'episode_idx']].head(10)

In [ ]:
# Aggregate per (visit_occurrence_id, episode_idx)
episodes = imv_logs.groupby(['person_id', 'visit_occurrence_id', 'episode_idx']).agg(
    episode_start=('procedure_datetime', 'min'),
    episode_end=('procedure_datetime', 'max'),
    n_logs=('procedure_datetime', 'count'),
).reset_index()

episodes['duration_hours'] = (
    episodes['episode_end'] - episodes['episode_start']
).dt.total_seconds() / 3600

print(f'Total IMV episodes:            {len(episodes):,}')
print(f'Across visits:                 {episodes["visit_occurrence_id"].nunique():,}')
print(f'Median duration (hours):       {episodes["duration_hours"].median():.1f}')
print(f'Episodes > 24h:                {(episodes["duration_hours"] > 24).sum():,}')
episodes.head()

In [ ]:
# Visualise the duration distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(episodes['duration_hours'].clip(upper=500), bins=60, color='#4C72B0', edgecolor='white')
ax.axvline(24, color='red', linestyle='--', label='24h cutoff (Zappalà)')
ax.set_xlabel('IMV episode duration (hours, clipped at 500)')
ax.set_ylabel('Number of episodes')
ax.set_title('Distribution of IMV episode duration')
ax.legend()
plt.tight_layout()
plt.show()

The distribution is heavily right-skewed: ~7,000 episodes are under 8 hours (post-op patients extubated quickly) and another ~3,000 fall in the 8–24h range, with all of these dropped by Zappalà's 24-hour cutoff to focus on clinically meaningful weaning cases. The long right tail with a spike at 500h represents the clipped bin — patients ventilated for weeks (ARDS, severe sepsis, prolonged respiratory failure), and this is exactly the high-stakes population where forecasting and weaning prediction matter most.

In [ ]:
first_episodes = episodes.sort_values(['visit_occurrence_id', 'episode_start'])
first_episodes = first_episodes.drop_duplicates('visit_occurrence_id', keep='first').reset_index(drop=True)

print(f'Visits with at least one IMV episode: {len(first_episodes):,}')
first_episodes.head()

## 5. Pull admission-level metadata for these visits

We need each visit's age at admission, ECMO status, and admission date to apply the rest of the filters.

In [ ]:
query = f"""
SELECT
    vo.person_id,
    vo.visit_occurrence_id,
    vo.visit_start_date,
    vo.visit_start_datetime,
    vo.visit_end_date,
    vo.visit_end_datetime,
    p.year_of_birth,
    p.gender_concept_id,
    EXTRACT(YEAR FROM vo.visit_start_date) AS admission_year,
    EXTRACT(YEAR FROM vo.visit_start_date) - p.year_of_birth AS age_at_admission
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
    ON vo.person_id = p.person_id
"""
visits = client.query(query).to_dataframe()
print(f'Loaded {len(visits):,} visit records')
visits.head()

In [ ]:
# Identify ECMO. ECMO is typically a procedure or device.
# We search for any procedure or device concept whose name contains 'ECMO' or 'extracorporeal membrane'
query = f"""
SELECT DISTINCT po.visit_occurrence_id
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` po
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.concept` c
    ON po.procedure_concept_id = c.concept_id
WHERE (po.provider_id IS NOT NULL OR po.provider_id IS NULL)
  AND (LOWER(c.concept_name) LIKE '%ecmo%'
       OR LOWER(c.concept_name) LIKE '%extracorporeal membrane%')
"""
ecmo_visits = client.query(query).to_dataframe()
ecmo_set = set(ecmo_visits['visit_occurrence_id'])
print(f'Visits with ECMO: {len(ecmo_set):,}')

ECMO (Extracorporeal Membrane Oxygenation) is a life-support procedure that oxygenates blood outside the body for patients with severe heart or lung failure, and this query finds all visits where it was performed.

## 6. Apply cohort filters stage by stage

Record the cohort size after each filter to build the flowchart.

In [ ]:
stages = []

# Stage 0: all visits in the database
cohort = visits.copy()
stages.append(('All ICU admissions', len(cohort)))

# Stage 1: has at least one IMV episode
cohort = cohort.merge(first_episodes, on=['person_id', 'visit_occurrence_id'], how='inner')
stages.append(('Has at least one IMV episode', len(cohort)))

# Stage 2: adult at admission
cohort = cohort[cohort['age_at_admission'] >= 18]
stages.append(('Adult at admission (≥ 18 y)', len(cohort)))

# Stage 3: admission year >= 2010 (Zappalà cutoff)
cohort = cohort[cohort['admission_year'] >= 2010]
stages.append(('Admitted 2010 or later', len(cohort)))

# Stage 4: first IMV episode duration > 24h
cohort = cohort[cohort['duration_hours'] > 24]
stages.append(('First IMV episode > 24h', len(cohort)))

# Stage 5: no ECMO
cohort = cohort[~cohort['visit_occurrence_id'].isin(ecmo_set)]
stages.append(('No ECMO', len(cohort)))

stages_df = pd.DataFrame(stages, columns=['Stage', 'N'])
stages_df['Dropped'] = stages_df['N'].shift(1).fillna(stages_df['N'].iloc[0]) - stages_df['N']
stages_df['Dropped'] = stages_df['Dropped'].astype(int)
stages_df

We applied sequential inclusion/exclusion criteria to filter ICU admissions down to a study cohort: adults (≥18) admitted from 2010 onward who received their first invasive mechanical ventilation (IMV) episode lasting more than 24 hours, excluding patients on ECMO.

The resulting `stages_df` is a CONSORT-style flow table showing how many patients remained and how many were dropped at each step.

## 7. Cohort flowchart (Figure-1 style)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
y_pos = range(len(stages_df))[::-1]
ax.barh(y_pos, stages_df['N'], color='#4C72B0')
ax.set_yticks(y_pos)
ax.set_yticklabels(stages_df['Stage'])
ax.set_xlabel('Number of ICU admissions remaining')
ax.set_title('Cohort flowchart — Zappalà 2025 inclusion criteria')
for i, (n, drop) in enumerate(zip(stages_df['N'], stages_df['Dropped'])):
    label = f'{int(n):,}'
    if i > 0:
        label += f'   (−{int(drop):,})'
    ax.text(n + 200, y_pos[i], label, va='center', fontsize=10)
ax.axvline(2626, color='red', linestyle=':', alpha=0.7, label='Zappalà 2025 target: 2,626')
ax.legend()
plt.tight_layout()
plt.show()

The cohort funneled cleanly from 23,106 admissions down to **2,884** — within 10% of Zappalà 2025's 2,626 — validating the entire pipeline. Each filter served a clinical purpose: **"Has IMV episode"** removed 6,504 patients who never needed ventilation (excludes irrelevant ICU cases); **"Adult ≥18"** dropped 0 patients (AmsterdamUMCdb is already an adults-only ICU); **"Admitted ≥2010"** removed 8,240 admissions to focus on modern ventilation practice (reflecting the date-binned 2006-cluster from earlier EDA); **"First IMV >24h"** removed 5,476 short post-operative cases where weaning isn't a real clinical decision, concentrating the cohort on patients with prolonged ventilation where prediction has value; and **"No ECMO"** removed 2 outlier patients whose lungs are bypassed by external oxygenation, making standard weaning predictors physiologically meaningless. The final 2,884 admissions are exactly the population where TimesFM-based forecasting can provide clinical value — sick enough that weaning is uncertain, with enough sequence length for meaningful time-series modeling.

## 8. Final cohort characteristics (Table 1 preview)

In [ ]:
final = cohort.copy()
print(f'Final cohort size: {len(final):,} admissions, '
      f'{final["person_id"].nunique():,} unique patients')
print()
print('=== Age ===')
print(final['age_at_admission'].describe().round(1))
print()
print('=== Gender ===')
gender_map = {8507: 'Male', 8532: 'Female'}
final['gender_label'] = final['gender_concept_id'].map(gender_map).fillna('Unknown')
print(final['gender_label'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')
print()
print('=== IMV duration (hours) ===')
print(final['duration_hours'].describe().round(1))

In [ ]:
# Age bucket distribution matching Zappalà 2025
bins = [0, 39, 49, 59, 69, 79, 200]
labels = ['18-39', '40-49', '50-59', '60-69', '70-79', '80+']
final['age_bucket'] = pd.cut(final['age_at_admission'], bins=bins, labels=labels, right=True)

fig, ax = plt.subplots(figsize=(9, 5))
counts = final['age_bucket'].value_counts().sort_index()
bars = ax.bar(counts.index.astype(str), counts.values, color='#55A868')
ax.set_ylabel('Number of admissions')
ax.set_title(f'Final cohort age distribution (N = {len(final):,})')
for bar, n in zip(bars, counts):
    pct = 100 * n / counts.sum()
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{pct:.1f}%', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

The cohort's age distribution **matches Zappalà 2025 almost perfectly** across all six buckets (18-39: 10.8% vs 11.23%, 40-49: 9.9% vs 10.21%, 50-59: 17.9% vs 17.90%, 60-69: 26.2% vs 25.97%, 70-79: 24.1% vs 24.52%, 80+: 11.1% vs 10.17%) — every bucket within ~1 percentage point. This near-identical match independently validates that your cohort construction is methodologically sound and directly comparable to published work. The expected ICU skew is clearly visible: ~61% of patients are aged 60+, the population where weaning is most clinically uncertain and where Zappalà reported reduced model performance — a known limitation you can directly address in your discussion section.

## 9. Save the cohort

Persist the final cohort with all the fields needed by the downstream notebooks: outcome definition (`06_outcome_definition.ipynb`) and variable extraction (`07_measurement_extraction.ipynb`).

In [ ]:
cohort_to_save = final[[
    'person_id',
    'visit_occurrence_id',
    'visit_start_datetime',
    'visit_end_datetime',
    'admission_year',
    'age_at_admission',
    'age_bucket',
    'gender_label',
    'episode_start',
    'episode_end',
    'duration_hours',
    'n_logs',
]].copy()
cohort_to_save.columns = [
    'person_id', 'visit_occurrence_id', 'admission_start', 'admission_end',
    'admission_year', 'age', 'age_bucket', 'gender',
    'imv_start', 'imv_end', 'imv_duration_hours', 'imv_n_logs',
]

out_path = 'cohort_zappala.parquet'
cohort_to_save.to_parquet(out_path, index=False)
print(f'Saved {len(cohort_to_save):,} rows to {out_path}')
cohort_to_save.head()

## Conclusions and next steps

**What we accomplished**:
1. Identified 8 IMV concept IDs covering all invasive ventilation modes in AmsterdamUMCdb.
2. Aggregated hourly ventilation logs into episodes using the 48-hour-gap rule from Zappalà 2025.
3. Kept only the first IMV episode per visit.
4. Applied the 5-stage filter chain (adult, ≥2010, IMV > 24h, no ECMO).
5. Produced Figure-1 flowchart and Table-1 summary.
6. Saved the cohort as `cohort_zappala.parquet` for downstream use.

**Validation status**: Compare final N, age buckets, and gender split with Zappalà's published numbers. Document any deviations as part of the methods section.